# Fine-tune Zipformer-30M-RNNT (hynt) trên dữ liệu trợ lý ảo

Pipeline: **TorchScript → icefall `AsrModel` → fine-tune (pruned RNN-T, k2) → export ONNX → sherpa-onnx**.

| | |
|---|---|
| Model gốc | `hynt/Zipformer-30M-RNNT-6000h` — Zipformer2 non-causal, cấu hình *small* (6 stack × 2 layer), BPE 2000 |
| Dữ liệu | `train.jsonl`: 1545 câu query **+ 3491 lượt user trong `history`** (có audio + `audio_text`) ≈ 5036 câu |
| Chia | theo **hội thoại** (cùng người nói không lọt giữa train/dev). Dev = 200 hội thoại (seed 0, đủ mọi lượt), test = 10 query — query của dev/test **giống hệt** thí nghiệm hotword trước |
| Giọng | gán nhãn Bắc/Trung/Nam theo hội thoại bằng từ đặc trưng; **oversample giọng Trung** (`OVERSAMPLE`) với speed perturb nhiều mức + nhiễu/gain trên feature; WER dev in **riêng từng giọng** |
| Phần cứng | GPU (Colab/Kaggle T4 là đủ, ~23 h audio/epoch ≈ 10–15 phút/epoch). CPU chạy được nhưng chậm |
| Baseline (sherpa greedy, **int8**, WER thô) | dev-query **0.2089**, test **0.1429** — cell 11 đo lại cả fp32 lẫn int8 |

> ⚠️ **License `cc-by-nc-nd-4.0`**: không dùng thương mại; **không được chia sẻ/công bố weights đã fine-tune** (ND). Chỉ dùng nội bộ.

Thứ tự chạy: từ trên xuống. Cell 5 có `assert` kiểm tra weights nạp đúng — nếu fail thì dừng lại, đừng train.

## 1. Cấu hình

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/train")   # thư mục chứa train.jsonl, audio/ (Kaggle: /kaggle/input/<dataset>/train)
WORK     = Path("/content/zf_ft")   # features, checkpoint, ONNX (Kaggle: /kaggle/working/zf_ft)
HF_REPO  = "hynt/Zipformer-30M-RNNT-6000h"
MODEL_OUT = WORK / "zipformer-ft"   # thư mục model ONNX cuối — trỏ pipeline vào đây
ICEFALL_COMMIT = "9c966762e058bd7ba79d9da71e42fc06b64b703e"   # bản đã đối chiếu API

SEED, DEV_N = 0, 200
TEST_IDS = ["train_0001", "train_0002", "train_0003", "train_0004", "train_0005",
            "train_0006", "train_0007", "train_0008", "train_0009", "train_0011"]
CODE_DIR = None           # repo voice-agent-pipeline (cần textnorm/), vd "/kaggle/input/zai-code/ZAI_TEST"
TN_BY_HYP = True          # nhãn: số/ký hiệu đọc theo cách người nói thật (chọn theo hyp model gốc), cần CODE_DIR
EXTRA_JSONL = None        # (tuỳ chọn) dữ liệu tổng quát chống "quên": mỗi dòng {"audio": "/abs/x.wav", "text": "..."}

USE_HISTORY    = True         # thêm các lượt user trong history (×3.3 dữ liệu)
SPEED_PERTURB  = (0.9, 1.1)   # () để tắt. Bắc/Nam: bản gốc + các mức này
SPEED_TRUNG    = (0.85, 0.9, 0.95, 1.05, 1.1, 1.15)   # giọng Trung: nhiều mức hơn để các bản lặp nghe khác nhau
OVERSAMPLE     = {"Bắc": 1, "Nam": 1, "Trung": 3, "Không rõ": 1}   # hệ số so với Bắc (1 = bản gốc + SPEED_PERTURB)
FEAT_AUG_P     = 0.5          # xác suất thêm nhiễu (SNR 10–30 dB) + đổi gain (±6 dB) trên fbank mỗi câu train; 0 để tắt
SELECT_METRIC  = "macro"      # chọn epoch theo: "macro" = TB WER Bắc/Nam/Trung (Trung nặng ký hơn), "all" = WER gộp
EPOCHS         = 15
BASE_LR        = 0.002        # icefall finetune mặc định 0.0045 (cho ~1000 h); 3 h dữ liệu -> thấp hơn
LR_EPOCHS      = 6.0          # Eden: lr giảm theo epoch
LR_BATCHES     = 5000.0
WARMUP_BATCHES = 100
MAX_DURATION   = 300          # giây audio / batch. T4 16GB: 300–400; CPU: 100
NUM_WORKERS    = 2
EARLY_STOP     = 4            # dừng nếu dev WER không cải thiện sau N epoch
AVG_TOP_K      = 3            # trung bình weights của K epoch tốt nhất

WORK.mkdir(parents=True, exist_ok=True)
EXP = WORK / "exp"; EXP.mkdir(exist_ok=True)

**(Colab)** đưa dữ liệu lên: nén thư mục `train` thành zip, để trên Google Drive rồi giải nén:

In [ ]:
# from google.colab import drive; drive.mount("/content/drive")
# !unzip -q "/content/drive/MyDrive/train.zip" -d /content/
assert (DATA_DIR / "train.jsonl").exists(), f"Không thấy {DATA_DIR/'train.jsonl'}"

## 2. Cài đặt (k2 tự chọn wheel khớp torch/CUDA đang có)

In [ ]:
import sys, subprocess, re, urllib.request, torch

def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
pip("lhotse", "lilcom", "sentencepiece", "sherpa-onnx", "jiwer", "kaldialign", "pypinyin", "tensorboard",
    "onnx", "onnxruntime", "onnxmltools", "soundfile", "huggingface_hub", "dill", "typeguard")

try:
    import k2  # noqa
except ImportError:
    tv, cu = torch.__version__.split("+")[0], torch.version.cuda
    py = f"cp{sys.version_info.major}{sys.version_info.minor}"
    idx = "https://k2-fsa.github.io/k2/cuda.html" if cu else "https://k2-fsa.github.io/k2/cpu.html"
    tag = f"+cuda{cu}.torch{tv}-" if cu else f"+cpu.torch{tv}-"
    html = urllib.request.urlopen(idx).read().decode()
    urls = [u for u in set(re.findall(r'href="([^"]+\.whl)"', html))
            if tag in u and f"-{py}-{py}-" in u and "linux" in u and "x86_64" in u]
    if not urls:
        raise RuntimeError(f"Không có wheel k2 cho torch {tv} / CUDA {cu} / {py}. Cài torch khớp một wheel trong {idx}, "
                           "vd: pip install torch==2.12.1 --index-url https://download.pytorch.org/whl/cu126 rồi restart kernel.")
    url = max(urls, key=lambda u: re.search(r"dev(\d+)", u).group(1))
    print("k2 wheel:", url.rsplit("/", 1)[-1]); pip(url)
import k2
print("k2 OK, cuda:", k2.with_cuda if hasattr(k2, "with_cuda") else "?")

In [ ]:
import os
ICEFALL = WORK / "icefall"
if not ICEFALL.exists():
    subprocess.check_call(["git", "clone", "-q", "https://github.com/k2-fsa/icefall", str(ICEFALL)])
    subprocess.check_call(["git", "-C", str(ICEFALL), "checkout", "-q", ICEFALL_COMMIT])
ZDIR = ICEFALL / "egs/librispeech/ASR/zipformer"
for p in (str(ZDIR), str(ICEFALL)):
    if p not in sys.path: sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{ICEFALL}:{os.environ.get('PYTHONPATH', '')}"

## 3. Text normalization (viết → đọc): chữ HOA, bỏ dấu câu, số thành chữ, số điện thoại đọc từng chữ số

In [ ]:
%%writefile {WORK}/tn_vi.py
"""Text normalization (written -> spoken) cho nhãn train Zipformer: chữ HOA, không dấu câu, số đọc thành chữ."""
import re

DIG = ["không", "một", "hai", "ba", "bốn", "năm", "sáu", "bảy", "tám", "chín"]


def _read3(n, full):
    """0..999. full=True: đọc đủ 'không trăm', 'lẻ' (dùng cho nhóm không đứng đầu)."""
    h, t, u = n // 100, n // 10 % 10, n % 10
    out = []
    if h or full:
        out += [DIG[h], "trăm"]
    if t == 0:
        if u and (h or full):
            out.append("lẻ")
    elif t == 1:
        out.append("mười")
    else:
        out += [DIG[t], "mươi"]
    if u:
        if u == 1 and t >= 2:
            out.append("mốt")
        elif u == 5 and t >= 1:
            out.append("lăm")
        else:
            out.append(DIG[u])
    return out


def read_number(n: int) -> str:
    if n == 0:
        return "không"
    units = ["", "nghìn", "triệu", "tỷ"]
    groups = []
    while n:
        groups.append(n % 1000)
        n //= 1000
    out = []
    for i in range(len(groups) - 1, -1, -1):
        g = groups[i]
        if g == 0:
            continue
        out += _read3(g, full=i < len(groups) - 1)
        if i:
            out.append(units[i])
    return " ".join(out)


def read_digits(s: str) -> str:
    return " ".join(DIG[int(c)] for c in s)


def _num(s: str) -> str:
    """chuỗi chữ số -> đọc. Bắt đầu bằng 0 (0983, số điện thoại) hoặc > 12 chữ số -> đọc từng chữ số."""
    if (s.startswith("0") and len(s) > 1) or len(s) > 12:
        return read_digits(s)
    return read_number(int(s))


def normalize(text: str) -> str:
    t = text
    # số điện thoại tách nhóm: "0983 214 6755" -> gộp rồi đọc từng chữ số
    t = re.sub(r"\b0\d{2,4}(?:[ .]\d{3,4}){1,3}\b", lambda m: read_digits(re.sub(r"\D", "", m.group())), t)
    t = re.sub(r"(\d+):(\d{2})", lambda m: f"{_num(m[1])} giờ {_num(m[2])}", t)          # 7:30
    t = re.sub(r"(\d+)\s*h\b", lambda m: f"{_num(m[1])} giờ", t)                         # 24h, 21h
    t = re.sub(r"(\d+)\s*%", lambda m: f"{_num(m[1])} phần trăm", t)
    t = re.sub(r"\b[1-9]\d{0,2}(?:\.\d{3})+\b", lambda m: read_number(int(m.group().replace(".", ""))), t)  # 50.000
    t = re.sub(r"(\d+),(\d+)", lambda m: f"{_num(m[1])} phẩy {read_digits(m[2]) if m[2].startswith('0') else _num(m[2])}", t)  # 1,5
    t = re.sub(r"(?<=[A-Za-zÀ-ỹ])(?=\d)|(?<=\d)(?=[A-Za-zÀ-ỹ])", " ", t)                   # VTV1 -> VTV 1, E5
    t = re.sub(r"\d+", lambda m: f" {_num(m.group())} ", t)
    t = t.replace("+", " cộng ").replace("&", " và ").replace("@", " a còng ")
    t = re.sub(r"[^\w\s']", " ", t)  # bỏ dấu câu, '-', '…'
    t = t.replace("_", " ")
    return " ".join(t.upper().split())


if __name__ == "__main__":
    tests = {
        "Nạp 50 nghìn": "NẠP NĂM MƯƠI NGHÌN",
        "dưới 123 triệu thì 9 giờ": "DƯỚI MỘT TRĂM HAI MƯƠI BA TRIỆU THÌ CHÍN GIỜ",
        "số 0983 214 6755 nhé": "SỐ KHÔNG CHÍN TÁM BA HAI MỘT BỐN SÁU BẢY NĂM NĂM NHÉ",
        "gọi 09889990002": "GỌI KHÔNG CHÍN TÁM TÁM CHÍN CHÍN CHÍN KHÔNG KHÔNG KHÔNG HAI",
        "VTV1, E5, RON 95": "VTV MỘT E NĂM RON CHÍN MƯƠI LĂM",
        "Góc Phố 24h.": "GÓC PHỐ HAI MƯƠI BỐN GIỜ",
        "105 và 21 và 1005 và 2015": "MỘT TRĂM LẺ NĂM VÀ HAI MƯƠI MỐT VÀ MỘT NGHÌN KHÔNG TRĂM LẺ NĂM VÀ HAI NGHÌN KHÔNG TRĂM MƯỜI LĂM",
        "50.000 đồng, 1,5 lít": "NĂM MƯƠI NGHÌN ĐỒNG MỘT PHẨY NĂM LÍT",
        "Omega-3 … S+": "OMEGA BA S CỘNG",
        "15 và 10 và 1000000": "MƯỜI LĂM VÀ MƯỜI VÀ MỘT TRIỆU",
    }
    bad = 0
    for i, exp in tests.items():
        got = normalize(i)
        if got != exp:
            bad += 1
            print("FAIL", repr(i), "->", got, "| expected", exp)
    print(f"{len(tests) - bad}/{len(tests)} ok")

In [ ]:
sys.path.insert(0, str(WORK))
import importlib, tn_vi; importlib.reload(tn_vi)
from tn_vi import normalize
for s in ["Nạp 50 nghìn cho số 0983 214 6755", "dưới 123 triệu thì 9 giờ", "VTV1, E5, RON 95, Góc Phố 24h"]:
    print(s, "->", normalize(s))

## 4. Tải model, tokens

In [ ]:
import shutil
import sentencepiece as spm
from huggingface_hub import snapshot_download

MD = Path(snapshot_download(HF_REPO, allow_patterns=["*.onnx", "bpe.model", "jit_script.pt"]))
sp = spm.SentencePieceProcessor(model_file=str(MD / "bpe.model"))
assert sp.get_piece_size() == 2000 and sp.id_to_piece(0) == "<blk>"
LANG = WORK / "lang"; LANG.mkdir(exist_ok=True)
shutil.copy(MD / "bpe.model", LANG / "bpe.model")
(LANG / "tokens.txt").write_text("".join(f"{sp.id_to_piece(i)} {i}\n" for i in range(2000)), encoding="utf-8")
(LANG / "bpe.vocab").write_text("".join(f"{sp.id_to_piece(i)}\t{sp.get_score(i)}\n" for i in range(2000)), encoding="utf-8")
print(MD)

## 5. Dựng icefall `AsrModel` + nạp weights từ TorchScript

Kiến trúc suy ra từ shape tensor trong `jit_script.pt`. Key jit `encoder.encoder.*` → `encoder.*`, bỏ bản trùng `encoder.encoder_embed.*`.
Cell kiểm tra encoder output của model icefall **trùng** với bản jit.

In [ ]:
import json, random, logging
import soundfile as sf
from lhotse import Fbank, FbankConfig
from train import get_model, get_params, set_batch_count

ARCH = dict(num_encoder_layers="2,2,2,2,2,2", downsampling_factor="1,2,4,8,4,2",
            feedforward_dim="512,768,768,768,768,768", num_heads="4,4,4,8,4,4",
            encoder_dim="192,256,256,256,256,256", encoder_unmasked_dim="192,192,192,192,192,192",
            cnn_module_kernel="31,31,15,15,15,31", query_head_dim="32", value_head_dim="12",
            pos_head_dim="4", pos_dim=48, decoder_dim=512, joiner_dim=512, context_size=2,
            causal=False, chunk_size="16,32,64,-1", left_context_frames="64,128,256,-1")
params = get_params()
params.update(ARCH)
params.update(dict(vocab_size=2000, blank_id=0, use_transducer=True, use_ctc=False,
                   use_attention_decoder=False, use_cr_ctc=False))
model = get_model(params)

jit = torch.jit.load(str(MD / "jit_script.pt"), map_location="cpu").eval()
sd = {}
for k, v in jit.state_dict().items():
    if k.startswith("encoder.encoder_embed."):
        continue
    sd["encoder." + k[len("encoder.encoder."):] if k.startswith("encoder.encoder.") else k] = v
r = model.load_state_dict(sd, strict=False)
pnames = {n for n, _ in model.named_parameters()}
miss = [k for k in r.missing_keys if k in pnames]
print("missing params:", miss[:10], "| unexpected:", r.unexpected_keys[:10])
assert not miss and not r.unexpected_keys, "Kiến trúc không khớp — dừng lại."
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M params")

FBANK = Fbank(FbankConfig(num_mel_bins=80))
rows_all = [json.loads(l) for l in open(DATA_DIR / "train.jsonl", encoding="utf-8")]
x, srate = sf.read(DATA_DIR / rows_all[0]["audio"], dtype="float32")
feats = torch.from_numpy(FBANK.extract(x, srate))[None]
lens = torch.tensor([feats.shape[1]])
set_batch_count(model, 100000.0)
model.eval()
with torch.no_grad():
    a, al = jit.encoder(feats, lens)
    b, bl = model.forward_encoder(feats, lens)
diff = (a - b).abs().max().item()
print("encoder out", tuple(a.shape), "max|diff| =", diff)
assert a.shape == b.shape and diff < 1e-3, "Output lệch bản jit — kiểm tra lại ARCH."
del jit

## 6. Gán nhãn giọng, chia dữ liệu theo hội thoại, manifest lhotse, fbank

Nhãn giọng gán **theo hội thoại** (các lượt cùng một người nói) bằng từ đặc trưng có trọng số — chỉ là ước lượng
(≈ Bắc 46% / Nam 38% / Trung 15% số câu). Từ đa nghĩa (răng = khám răng, chi = chi phí) đã bỏ khỏi danh sách.

In [ ]:
from collections import Counter
from lhotse import CutSet, Recording, RecordingSet, SupervisionSegment, SupervisionSet, LilcomChunkyWriter
from lhotse.utils import fastcopy

DIALECT_WORDS = {
    "Trung": "rứa|ni|mô|chừ|hỉ|nớ|mần|mệ|răng rứa|bây chừ|chi rứa|làm chi|cái chi|chi mô|hè|ri|tê",
    "Nam":   "hông|nghen|nhen|hen|ổng|bả|dùm|nhiêu|vô|ghê|mắc|xài|lẹ|dzậy|vậy á|á|nha|nè|thiệt|bển|hổng|tui|mấy|"
             "chút xíu|mấy giờ|đặng|dzô|quá trời|rảnh|xíu",
    "Bắc":   "nhé|ạ|thế|nhỉ|ấy|đấy|vâng|hộ|giúp|bao giờ|chứ|nhá|cơ|đâu|à|nào|tớ|mình",
}
DIALECT_W = {"Trung": 3.0, "Nam": 1.5, "Bắc": 1.0}   # từ Trung hiếm và đặc trưng hơn
_dpat = {k: re.compile(r"(?<!\w)(?:" + v + r")(?!\w)") for k, v in DIALECT_WORDS.items()}


def dialect(texts):
    s = {k: DIALECT_W[k] * sum(len(p.findall(t.lower())) for t in texts) for k, p in _dpat.items()}
    best, tot = max(s, key=s.get), sum(s.values())
    return best if tot and s[best] / tot >= 0.5 else "Không rõ"


def utts_of(r):
    # mọi lượt user của một hội thoại: history trước, query cuối. id query = id sample (khớp TEST_IDS cũ)
    out = []
    if USE_HISTORY:
        for h in r["history"]:
            if h.get("role") == "user" and h.get("audio_text") and (DATA_DIR / h["audio"]).exists():
                out.append(dict(id=Path(h["audio"]).stem, audio=h["audio"], audio_text=h["audio_text"], query=False))
    if (DATA_DIR / r["audio"]).exists():
        out.append(dict(id=r["id"], audio=r["audio"], audio_text=r["audio_text"], query=True))
    d = dialect([u["audio_text"] for u in out])
    for u in out:
        u.update(sample=r["id"], dialect=d)
    return out


rows = [r for r in rows_all if (DATA_DIR / r["audio"]).exists()]
random.seed(SEED)
dev_samples = {r["id"] for r in random.sample([r for r in rows if r["id"] not in TEST_IDS], DEV_N)}   # như bản cũ
UTTS = [u for r in rows for u in utts_of(r)]
dev_rows = [u for u in UTTS if u["sample"] in dev_samples]
test_rows = [u for u in UTTS if u["sample"] in TEST_IDS and u["query"]]
train_rows = [u for u in UTTS if u["sample"] not in dev_samples and u["sample"] not in TEST_IDS]
REF = {u["id"]: u["audio_text"] for u in UTTS}
DIAL = {u["id"]: u["dialect"] for u in UTTS}
QUERY = {u["id"] for u in UTTS if u["query"]}
assert len(REF) == len(UTTS), "trùng id câu"
for name, us in [("train", train_rows), ("dev", dev_rows), ("test", test_rows)]:
    print(f"{name:5s} {len(us):5d} câu | " + ", ".join(f"{k} {v}" for k, v in Counter(u["dialect"] for u in us).most_common()))


# ---- nhãn TN theo cách đọc thật: tn_vi.normalize đọc số một kiểu chuẩn ("RON95" -> "chín mươi lăm", "24 độ" ->
# "hai mươi bốn") nhưng người nói hay đọc "chín lăm", "hai bốn" -> ~28% câu có số bị nhãn sai. Decode greedy model gốc
# trên các câu có số/ký hiệu rồi textnorm.vi_num.to_spoken chọn cách đọc khớp hyp nhất (S+ -> "s cộng", 2025 -> "hai không hai lăm").
LABEL = {u["id"]: normalize(u["audio_text"]) for u in UTTS}
if TN_BY_HYP:
    assert CODE_DIR, "TN_BY_HYP cần CODE_DIR (thư mục có textnorm/)"
    import sherpa_onnx, numpy as np
    sys.path.insert(0, str(CODE_DIR))
    from textnorm.vi_num import to_spoken
    from textnorm.textutil import norm as _norm
    _g = lambda n: str(next(p for p in sorted(MD.glob(f"{n}-*.onnx")) if not p.name.endswith(".int8.onnx")))
    _rec = sherpa_onnx.OfflineRecognizer.from_transducer(encoder=_g("encoder"), decoder=_g("decoder"), joiner=_g("joiner"),
        tokens=str(LANG / "tokens.txt"), num_threads=4, sample_rate=16000, feature_dim=80, decoding_method="greedy_search")
    need = [u for u in UTTS if re.search(r"[\d+&@%°]", u["audio_text"])]
    n_chg = 0
    for k in range(0, len(need), 16):
        ss = []
        for u in need[k:k + 16]:
            x, srate = sf.read(DATA_DIR / u["audio"], dtype="float32")
            s = _rec.create_stream(); s.accept_waveform(srate, np.ascontiguousarray(x)); ss.append(s)
        _rec.decode_streams(ss)
        for u, s in zip(need[k:k + 16], ss):
            lab = to_spoken(u["audio_text"], _norm(s.result.text))
            if lab:
                lab = " ".join(lab.upper().split())
                n_chg += lab != LABEL[u["id"]]
                LABEL[u["id"]] = lab
    del _rec
    print(f"TN theo hyp: {len(need)} câu có số/ký hiệu, đổi nhãn {n_chg} câu")
    for u in [u for u in need if LABEL[u["id"]] != normalize(u["audio_text"])][:5]:
        print("  ", u["audio_text"], "->", LABEL[u["id"]])


def make_cuts(items):
    # items: [(id, wav_path, raw_text)]
    recs, sups = [], []
    for i, wav, text in items:
        rec = Recording.from_file(wav, recording_id=i)
        if rec.num_channels > 1:
            raise ValueError(f"{wav}: cần mono")
        recs.append(rec)
        sups.append(SupervisionSegment(id=i, recording_id=i, start=0.0, duration=rec.duration, channel=0,
                                       text=LABEL.get(i) or normalize(text), language="vi"))
    return CutSet.from_manifests(recordings=RecordingSet.from_recordings(recs),
                                 supervisions=SupervisionSet.from_segments(sups)).to_eager()


CACHE_TAG = f"v3_tn{int(TN_BY_HYP)}_h{int(USE_HISTORY)}_" + "_".join(map(str, SPEED_TRUNG + SPEED_PERTURB))   # đổi cấu hình -> cache mới


def feats_for(cuts, name):
    out = WORK / f"cuts_{name}_{CACHE_TAG}.jsonl.gz"
    if out.exists():
        return CutSet.from_file(out)
    cuts = cuts.compute_and_store_features(extractor=FBANK, storage_path=str(WORK / f"feats_{name}_{CACHE_TAG}"),
                                           num_jobs=max(1, os.cpu_count() // 2), storage_type=LilcomChunkyWriter)
    cuts.to_file(out)
    return cuts


tr = [(u["id"], DATA_DIR / u["audio"], u["audio_text"]) for u in train_rows]
if EXTRA_JSONL:
    ex = [json.loads(l) for l in open(EXTRA_JSONL, encoding="utf-8")]
    tr += [(f"extra_{k:06d}", Path(e["audio"]), e["text"]) for k, e in enumerate(ex)]
    print("extra:", len(ex))
base = make_cuts(tr)
# mỗi câu: bản gốc + speed perturb (Trung dùng SPEED_TRUNG). Fbank chỉ tính 1 lần cho mỗi biến thể khác nhau
variants, orig = [], {}
for c in base:
    sid = c.supervisions[0].id
    for f in (1.0,) + (SPEED_TRUNG if DIAL.get(sid) == "Trung" else SPEED_PERTURB):
        v = c if f == 1.0 else c.perturb_speed(f)
        orig[v.id] = sid
        variants.append(v)
uniq = feats_for(CutSet.from_cuts(variants).filter(lambda c: 0.5 <= c.duration <= 30.0).to_eager(), "train")

# oversample: giọng d có OVERSAMPLE[d] × (1 + len(SPEED_PERTURB)) bản / câu; lặp vòng qua các biến thể speed
per = {}
for c in uniq:
    per.setdefault(orig.get(c.id, c.supervisions[0].id), []).append(c)
train_list = []
for sid, vs in per.items():
    want = max(1, round(OVERSAMPLE.get(DIAL.get(sid, "Không rõ"), 1) * (1 + len(SPEED_PERTURB))))
    train_list += [vs[k % len(vs)] if k < len(vs) else fastcopy(vs[k % len(vs)], id=f"{vs[k % len(vs)].id}_rep{k}")
                   for k in range(want)]
train_cuts = CutSet.from_cuts(train_list)
dev_cuts = feats_for(make_cuts([(u["id"], DATA_DIR / u["audio"], u["audio_text"]) for u in dev_rows]), "dev")
test_cuts = feats_for(make_cuts([(u["id"], DATA_DIR / u["audio"], u["audio_text"]) for u in test_rows]), "test")
hrs = Counter()
for c in train_cuts:
    hrs[DIAL.get(orig.get(c.id.split("_rep")[0], c.supervisions[0].id), "extra")] += c.duration / 3600
print(len(train_cuts), "train cuts / epoch,", f"{sum(hrs.values()):.2f} h |",
      ", ".join(f"{k} {v:.2f} h ({v / sum(hrs.values()):.0%})" for k, v in hrs.most_common()))

## 7. DataLoader, decode, WER

In [ ]:
import k2, jiwer
from torch.utils.data import DataLoader
from lhotse.dataset import K2SpeechRecognitionDataset, DynamicBucketingSampler, SimpleCutSampler, SpecAugment
from beam_search import greedy_search_batch

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FP16 = DEV.type == "cuda"
model.to(DEV)

import math


class FeatNoiseGain(torch.nn.Module):
    # augment trên log-mel (fbank đã tính sẵn nên không thêm nhiễu vào waveform được):
    # gain ±6 dB = cộng hằng số; nhiễu SNR 10–30 dB = log(exp(f) + n), n ngẫu nhiên theo từng frame/bin
    def __init__(self, p=0.5, gain_db=6.0, snr_db=(10.0, 30.0)):
        super().__init__(); self.p, self.gain_db, self.snr_db = p, gain_db, snr_db

    def forward(self, x, supervision_segments=None, *args, **kwargs):
        x = x.clone()
        for i in range(x.shape[0]):
            if random.random() >= self.p:
                continue
            f = x[i]
            valid = f.mean(-1) > -20.0                       # bỏ frame padding (LOG_EPSILON ≈ -23)
            p_sig = f[valid].exp().mean() if valid.any() else f.exp().mean()
            snr = random.uniform(*self.snr_db)
            noise = p_sig * 10 ** (-snr / 10) * 2 * torch.rand_like(f)
            g = random.uniform(-self.gain_db, self.gain_db) * math.log(10) / 10
            x[i] = torch.where(valid[:, None], torch.log(f.exp() + noise) + g, f)
        return x


train_tf = [SpecAugment(time_warp_factor=80, num_frame_masks=10, features_mask_size=27, num_feature_masks=2,
                        frames_mask_size=100)]
if FEAT_AUG_P > 0:
    train_tf.insert(0, FeatNoiseGain(FEAT_AUG_P))
train_sampler = DynamicBucketingSampler(train_cuts, max_duration=MAX_DURATION, shuffle=True, num_buckets=10,
                                        drop_last=True, seed=SEED)
train_dl = DataLoader(K2SpeechRecognitionDataset(return_cuts=True, input_transforms=train_tf),
                      sampler=train_sampler, batch_size=None, num_workers=NUM_WORKERS)

norm_raw = lambda s: " ".join(re.sub(r"[^\w\s]", " ", s.lower()).split())


@torch.no_grad()
def decode(cuts):
    model.eval()
    dl = DataLoader(K2SpeechRecognitionDataset(return_cuts=True),
                    sampler=SimpleCutSampler(cuts, max_duration=MAX_DURATION), batch_size=None, num_workers=NUM_WORKERS)
    out = {}
    for b in dl:
        x, xl = b["inputs"].to(DEV), b["supervisions"]["num_frames"].to(DEV)
        enc, encl = model.forward_encoder(x, xl)
        for c, h in zip(b["supervisions"]["cut"], greedy_search_batch(model=model, encoder_out=enc, encoder_out_lens=encl)):
            out[c.supervisions[0].id] = sp.decode(h).lower()
    return out


def wer(hyp):
    ids = sorted(hyp)
    spoken = jiwer.wer([(LABEL.get(i) or normalize(REF[i])).lower() for i in ids], [hyp[i] for i in ids])   # so với nhãn đã TN
    raw = jiwer.wer([norm_raw(REF[i]) for i in ids], [norm_raw(hyp[i]) for i in ids])     # như thí nghiệm trước
    return spoken, raw


def report(hyp):
    # WER spoken: gộp, chỉ query (so được với bản cũ), từng giọng, macro = TB Bắc/Nam/Trung
    r = {"all": wer(hyp)[0], "query": wer({i: h for i, h in hyp.items() if i in QUERY})[0]}
    for d in ("Bắc", "Nam", "Trung"):
        sub = {i: h for i, h in hyp.items() if DIAL.get(i) == d}
        if sub:
            r[d] = wer(sub)[0]
    r["macro"] = sum(r[d] for d in ("Bắc", "Nam", "Trung") if d in r) / max(1, sum(d in r for d in ("Bắc", "Nam", "Trung")))
    r["raw_query"] = wer({i: h for i, h in hyp.items() if i in QUERY})[1]
    return r


fmt = lambda r: " | ".join(f"{k} {r[k]:.4f}" for k in ("all", "macro", "Bắc", "Nam", "Trung", "query") if k in r)
h0 = decode(dev_cuts)
base_dev = report(h0)
base_test = wer(decode(test_cuts))
print("PRETRAINED dev  ", fmt(base_dev))
print(f"PRETRAINED dev raw-query {base_dev['raw_query']:.4f}   (nên gần 0.209 của sherpa greedy int8; lệch nhiều => feature/kiến trúc có vấn đề)")
print(f"PRETRAINED test  spoken {base_test[0]:.4f} raw {base_test[1]:.4f}")

## 8. Fine-tune

In [ ]:
import math, time, copy
from optim import ScaledAdam, Eden
from icefall.utils import get_parameter_groups_with_lrs, create_grad_scaler, torch_autocast

torch.manual_seed(SEED)
opt = ScaledAdam(get_parameter_groups_with_lrs(model, lr=BASE_LR, include_names=True), lr=BASE_LR, clipping_scale=2.0)
sch = Eden(opt, LR_BATCHES, LR_EPOCHS, warmup_batches=WARMUP_BATCHES)
scaler = create_grad_scaler(enabled=FP16, init_scale=1.0)

history, step, best, bad_epochs = [], 0, (base_dev[SELECT_METRIC], 0), 0
torch.save({"model": model.state_dict()}, EXP / "epoch-0.pt")
for epoch in range(1, EPOCHS + 1):
    sch.step_epoch(epoch - 1)
    train_sampler.set_epoch(epoch)
    model.train()
    t0, tot_loss, tot_frames, skipped = time.time(), 0.0, 0, 0
    for b in train_dl:
        step += 1
        set_batch_count(model, 100000.0 + step * MAX_DURATION / 600)  # giữ ScheduledFloat ở giá trị cuối (như finetune.py)
        x, xl = b["inputs"].to(DEV), b["supervisions"]["num_frames"].to(DEV)
        y = k2.RaggedTensor(sp.encode(b["supervisions"]["text"], out_type=int))
        with torch_autocast(enabled=FP16):
            simple, pruned, _, _, _ = model(x=x, x_lens=xl, y=y, prune_range=5, am_scale=0.0, lm_scale=0.25)
            loss = 0.5 * simple + pruned
        if not torch.isfinite(loss):
            skipped += 1; opt.zero_grad(); continue
        scaler.scale(loss).backward()
        sch.step_batch(step)
        scaler.step(opt); scaler.update(); opt.zero_grad()
        if FP16 and step % 100 == 0:  # như icefall: đừng để grad scale tụt quá thấp
            s = scaler._scale.item()
            if s < 8.0 or (s < 32.0 and step % 400 == 0):
                scaler.update(s * 2.0)
        tot_loss += loss.item(); tot_frames += (xl // 4).sum().item()
    rd = report(decode(dev_cuts))
    torch.save({"model": model.state_dict()}, EXP / f"epoch-{epoch}.pt")
    history.append(dict(epoch=epoch, loss=tot_loss / max(tot_frames, 1), **rd))
    print(f"epoch {epoch:2d} | loss/frame {tot_loss / max(tot_frames, 1):.4f} | lr {sch.get_last_lr()[0]:.2e} | "
          f"{fmt(rd)} | {time.time() - t0:.0f}s" + (f" | skipped {skipped}" if skipped else ""))
    if rd[SELECT_METRIC] < best[0] - 1e-4:
        best, bad_epochs = (rd[SELECT_METRIC], epoch), 0
    else:
        bad_epochs += 1
        if bad_epochs >= EARLY_STOP:
            print("early stop"); break
print("best epoch", best[1], f"dev {SELECT_METRIC}", round(best[0], 4))
print("pretrained:", fmt(base_dev))

## 9. Trung bình top-K checkpoint, chọn model cuối

In [ ]:
def load(ep):
    return torch.load(EXP / f"epoch-{ep}.pt", map_location="cpu", weights_only=False)["model"]

top = [h["epoch"] for h in sorted(history, key=lambda h: h[SELECT_METRIC])[:AVG_TOP_K]]
cands = {f"epoch {best[1]}": load(best[1])}
if len(top) > 1:
    sds = [load(e) for e in top]
    avg = {k: (sum(s[k].double() for s in sds) / len(sds)).to(sds[0][k].dtype) if sds[0][k].is_floating_point() else sds[0][k]
           for k in sds[0]}
    cands[f"avg {sorted(top)}"] = avg
results = {}
for name, s in cands.items():
    model.load_state_dict(s)
    results[name] = (report(decode(dev_cuts)), s)
    print(f"{name:20s} {fmt(results[name][0])}")
final = min(results, key=lambda n: results[n][0][SELECT_METRIC])
model.load_state_dict(results[final][1])
torch.save({"model": model.state_dict()}, EXP / "epoch-99.pt")   # tên này để export-onnx.py --epoch 99 --avg 1
ft_dev, ft_test = report(decode(dev_cuts)), wer(decode(test_cuts))
print(f"\nCHỌN: {final}   (WER spoken trên dev)")
print(f"{'':11s}" + "".join(f"{k:>9s}" for k in ("all", "macro", "Bắc", "Nam", "Trung", "query")) + "   test spoken/raw")
for name, r, t in [("pretrained", base_dev, base_test), ("finetuned", ft_dev, ft_test)]:
    print(f"{name:11s}" + "".join(f"{r.get(k, float('nan')):9.4f}" for k in ("all", "macro", "Bắc", "Nam", "Trung", "query"))
          + f"   {t[0]:.4f} / {t[1]:.4f}")

In [ ]:
# so từng câu test
h_ft = decode(test_cuts)
for i in sorted(h_ft):
    print(i, "\n  REF:", REF[i], "\n  FT :", h_ft[i])

## 10. Export ONNX (fp32 + int8) cho sherpa-onnx

In [ ]:
flags = []
for k, v in ARCH.items():
    flags += [f"--{k.replace('_', '-')}", str(v)]
cmd = [sys.executable, str(ZDIR / "export-onnx.py"), "--tokens", str(LANG / "tokens.txt"), "--exp-dir", str(EXP),
       "--epoch", "99", "--avg", "1", "--use-averaged-model", "0", "--enable-int8-quantization", "1", *flags]
print(" ".join(cmd))
subprocess.run(cmd, cwd=ZDIR, check=True, env={**os.environ, "PYTHONPATH": f"{ICEFALL}:{ZDIR}"})

MODEL_OUT.mkdir(parents=True, exist_ok=True)
for f in EXP.glob("*-epoch-99-avg-1*.onnx"):
    shutil.copy(f, MODEL_OUT / f.name)
for f in ("tokens.txt", "bpe.model", "bpe.vocab"):
    shutil.copy(LANG / f, MODEL_OUT / f)
print("MODEL_OUT =", MODEL_OUT)
print(sorted(p.name for p in MODEL_OUT.iterdir()))

## 11. Kiểm tra bản ONNX bằng sherpa-onnx (greedy), so với model gốc

In [ ]:
import numpy as np, sherpa_onnx

def sherpa_wer(d, items, int8=False):
    # chọn file tường minh: glob "encoder-*.onnx" cũng khớp "...int8.onnx"
    g = lambda n: str(next(p for p in sorted(Path(d).glob(f"{n}-*.onnx")) if p.name.endswith(".int8.onnx") == int8))
    rec = sherpa_onnx.OfflineRecognizer.from_transducer(encoder=g("encoder"), decoder=g("decoder"), joiner=g("joiner"),
        tokens=str(LANG / "tokens.txt"), num_threads=4, sample_rate=16000, feature_dim=80, decoding_method="greedy_search")
    hyp = {}
    for k in range(0, len(items), 16):
        ss = []
        for r in items[k:k + 16]:
            x, srate = sf.read(DATA_DIR / r["audio"], dtype="float32")
            s = rec.create_stream(); s.accept_waveform(srate, np.ascontiguousarray(x)); ss.append(s)
        rec.decode_streams(ss)
        hyp.update({r["id"]: s.result.text.strip().lower() for r, s in zip(items[k:k + 16], ss)})
    return hyp

for name, d in [("gốc", MD), ("fine-tuned", MODEL_OUT)]:
    for q in (False, True):
        dv, ts = report(sherpa_wer(d, dev_rows, q)), wer(sherpa_wer(d, test_rows, q))
        print(f"{name:10s} {'int8' if q else 'fp32'}  {fmt(dv)} | dev raw-query {dv['raw_query']:.4f} | test raw {ts[1]:.4f}")

## Dùng trong pipeline

Model nằm sẵn ở thư mục `MODEL_OUT` (in ra ở cell 10). Truyền thẳng đường dẫn đó:

```python
from asr_zipformer import ZipformerASR
asr = ZipformerASR(str(MODEL_OUT))     # thư mục local thay cho HF id
```
hoặc `python pipeline.py ... --asr-model zipformer --zipformer <MODEL_OUT> --bias`.

Hotword biasing vẫn dùng được với model mới. Nên chạy lại phần tune hotword trên dev, vì ngưỡng score có thể đã khác sau fine-tune.

**Nếu muốn tốt hơn nữa**
- `EXTRA_JSONL`: trộn thêm khoảng 5–10 h dữ liệu tổng quát (FLEURS-vi, Common Voice vi) để model không quên kiến thức cũ.
- Thử `BASE_LR` trong khoảng 0.001–0.0045 và `EPOCHS` 10–30, chọn theo dev.
- Đánh giá cuối trên `public_test`, đừng chỉ dựa vào dev: câu train lặp mẫu nhiều, dễ overfit.